In [8]:
import requests
from lxml import html
from tqdm import tqdm
import os

BASE = "https://23fbuscador.rtve.es"

headers = {
    "User-Agent": "Mozilla/5.0"
}

def get_document_links(page_number):
    url = f"{BASE}/?page_size=200&page={page_number}"
    r = requests.get(url, headers=headers)
    tree = html.fromstring(r.content)

    links = tree.xpath("//table/tbody/tr/td/a/@href")
    return [BASE + link for link in links]


def get_total_pages():
    r = requests.get(f"{BASE}/?page_size=200&page=1", headers=headers)
    tree = html.fromstring(r.content)

    # Extract "Página X de Y"
    text = tree.xpath("//span[@class='nav-position']/text()")
    if not text:
        return 1

    # Example: "Página 1 de 7"
    parts = text[0].split()
    return int(parts[-1])  # last number is total pages


def download_transcript(url, title):
    r = requests.get(url, headers=headers)
    tree = html.fromstring(r.content)

    pre = tree.xpath("//pre/text()")
    if not pre:
        return None

    return pre[0]


def scrape_all():
    os.makedirs("transcripts", exist_ok=True)

    total_pages = get_total_pages()
    print(f"Total pages: {total_pages}")

    all_links = []

    # Collect all document links
    for page in range(1, total_pages + 1):
        links = get_document_links(page)
        all_links.extend(links)

    print(f"Found {len(all_links)} documents")

    # Download each transcript
    for url in tqdm(all_links):
        # Extract title from URL (last part)
        doc_id = url.split("/")[-1].split("?")[0]
        title = f"document_{doc_id}"

        text = download_transcript(url, title)
        if not text:
            print(f"⚠️ No transcript found for {url}")
            continue

        filename = f"transcripts/{title}.txt"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(text)

    print("Done! All transcripts saved in /transcripts")

In [9]:
scrape_all()

Total pages: 1
Found 167 documents


100%|██████████| 167/167 [00:55<00:00,  2.99it/s]

Done! All transcripts saved in /transcripts


In [12]:
import requests
from lxml import html
from tqdm import tqdm
import pandas as pd
import os
import hashlib

BASE = "https://23fbuscador.rtve.es"
headers = {"User-Agent": "Mozilla/5.0"}


# -----------------------------
# Helper: Get total number of pages
# -----------------------------
def get_total_pages():
    r = requests.get(f"{BASE}/?page_size=200&page=1", headers=headers)
    tree = html.fromstring(r.content)

    text = tree.xpath("//span[@class='nav-position']/text()")
    if not text:
        return 1

    # Example: "Página 1 de 7"
    parts = text[0].split()
    return int(parts[-1])


# -----------------------------
# Helper: Extract document rows from a listing page
# -----------------------------
def get_page_documents(page_number):
    url = f"{BASE}/?page_size=200&page={page_number}"
    r = requests.get(url, headers=headers)
    tree = html.fromstring(r.content)

    rows = tree.xpath("//table/tbody/tr")

    docs = []
    for row in rows:
        title = row.xpath(".//td[1]/a/text()")
        href = row.xpath(".//td[1]/a/@href")
        pages = row.xpath(".//td[2]/text()")
        summary = row.xpath(".//td[4]/text()")
        tags = row.xpath(".//td[5]//span[@class='tag-chip']/text()")

        if not href:
            continue

        docs.append({
            "title": title[0].strip() if title else "",
            "url": BASE + href[0],
            "pages": pages[0].strip() if pages else None,
            "summary": summary[0].strip() if summary else "",
            "tags": tags,
        })

    return docs


# -----------------------------
# Helper: Download transcript text from a document page
# -----------------------------
def download_transcript(url):
    r = requests.get(url, headers=headers)
    tree = html.fromstring(r.content)

    pre = tree.xpath("//pre/text()")
    if not pre:
        return None

    return pre[0]


# -----------------------------
# Helper: Generate safe filename
# -----------------------------
def make_safe_filename(title):
    # Remove illegal characters
    cleaned = "".join(c for c in title if c.isalnum() or c in " _-").strip()

    # If too long, shorten and add a hash
    if len(cleaned) > 100:
        hash_suffix = hashlib.md5(cleaned.encode()).hexdigest()[:8]
        cleaned = cleaned[:80] + "_" + hash_suffix

    return cleaned + ".txt"


# -----------------------------
# Main scraper
# -----------------------------
def scrape_all():
    os.makedirs("transcripts", exist_ok=True)

    total_pages = get_total_pages()
    print(f"Total pages: {total_pages}")

    all_docs = []

    # Step 1: Collect metadata for all documents
    for page in range(1, total_pages + 1):
        docs = get_page_documents(page)
        all_docs.extend(docs)

    print(f"Found {len(all_docs)} documents")

    # Step 2: Download transcripts + save .txt files
    for doc in tqdm(all_docs):
        transcript = download_transcript(doc["url"])
        doc["transcript"] = transcript

        if transcript:
            filename = make_safe_filename(doc["title"])
            filepath = os.path.join("transcripts", filename)

            with open(filepath, "w", encoding="utf-8") as f:
                f.write(transcript)

            doc["filename"] = filename
        else:
            doc["filename"] = None

    # Step 3: Convert to DataFrame
    df = pd.DataFrame(all_docs)
    return df

Total pages: 1
Found 167 documents


100%|██████████| 167/167 [00:03<00:00, 47.07it/s]


,title,url,pages,summary,tags,transcript,filename
0,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1860?...,3,El juicio oral 2/81 celebrado en febrero de 19...,"[C/SG/2820/20-02-82, DTOR., Vista oral 2/81]",C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\...,Vista oral 281 del Consejo Supremo de Justicia...
1,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1859?...,4,Resumen global del documento:\n\nEl documento ...,"[C/SG/2896/22-02-82, Vista oral 2/81, Consejo ...",C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...
2,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1858?...,5,Resumen global del documento:\n\nEl documento ...,"[C/SG/2992/24-02-82, Vista Oral 2/81, Consejo ...",C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...
3,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1857?...,6,El documento recoge el desarrollo de la sesión...,"[C/SG/3.081/25-02-82, Vista Oral 2/81, Consejo...",C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nA...,Vista oral 281 del Consejo Supremo de Justicia...
4,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1856?...,6,Resumen global del documento sobre la sesión d...,"[C/SG/3.249/26-02-82, SG, Consejo Supremo de J...",C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\...,Vista oral 281 del Consejo Supremo de Justicia...


In [11]:
# Run scraper
df = scrape_all()
df.head()

Total pages: 1
Found 167 documents


 56%|█████▌    | 93/167 [00:01<00:01, 59.40it/s]


OSError: [Errno 36] File name too long: 'transcripts/Informe de las distintas Jefaturas Superiores Comisaría General de Información Situación actual en las distintas regiones policiales y acciones de protesta previstas en relación a la ocupación del Palacio del Congreso de los Diputados 26 de febrero de 1981.txt'

In [14]:
df.head()

,title,url,pages,summary,tags,transcript,filename
0,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1860?...,3,El juicio oral 2/81 celebrado en febrero de 19...,"[C/SG/2820/20-02-82, DTOR., Vista oral 2/81]",C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\...,Vista oral 281 del Consejo Supremo de Justicia...
1,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1859?...,4,Resumen global del documento:\n\nEl documento ...,"[C/SG/2896/22-02-82, Vista oral 2/81, Consejo ...",C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...
2,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1858?...,5,Resumen global del documento:\n\nEl documento ...,"[C/SG/2992/24-02-82, Vista Oral 2/81, Consejo ...",C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...
3,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1857?...,6,El documento recoge el desarrollo de la sesión...,"[C/SG/3.081/25-02-82, Vista Oral 2/81, Consejo...",C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nA...,Vista oral 281 del Consejo Supremo de Justicia...
4,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1856?...,6,Resumen global del documento sobre la sesión d...,"[C/SG/3.249/26-02-82, SG, Consejo Supremo de J...",C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\...,Vista oral 281 del Consejo Supremo de Justicia...
